# MCP - Hands-On Tour

A single notebook walking through `../docs/`: build a server &rarr; connect
a client &rarr; bridge it to a local model &rarr; explore resources,
errors, schemas, authorization, and sampling. Everything runs against a
**real local MCP server you write in this notebook** and, where a model
is needed, a **local Ollama model** - no hosted API, no signup.

Every code block here was actually run while building this module -
including one documented failure (an async pytest fixture pitfall,
section 8) that's worth seeing fail before seeing the fix.

## Setup

```bash
ollama pull llama3.1:8b
pip install mcp openai pydantic
```

Jupyter supports top-level `await` directly in cells - no
`asyncio.run()` wrapper needed below, unlike the standalone scripts in
`../examples/`.

## 1. Write a server to a file

MCP servers run as separate processes (see `docs/04-Transports-stdio-and-HTTP.md`),
so we write it to disk first, then spawn it from the client cells below.

In [ ]:
server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("infra-tools")

@mcp.tool()
def get_disk_usage(hostname: str) -> dict:
    "Get current disk usage percentage for a given host."
    known_hosts = {"db-primary-01": 92, "web-node-01": 41}
    if hostname not in known_hosts:
        return {"error": True, "message": f"Unknown hostname: {hostname!r}. Known hosts: {list(known_hosts)}"}
    return {"hostname": hostname, "disk_percent": known_hosts[hostname]}

@mcp.resource("runbook://crashloop")
def crashloop_runbook() -> str:
    "Runbook for debugging CrashLoopBackOff."
    return "Check kubectl describe pod and kubectl logs --previous."

@mcp.prompt()
def incident_summary(log_text: str) -> str:
    "Build a prompt that summarizes an incident log."
    return f"Summarize this incident log in 2 sentences: {log_text}"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("notebook_server.py", "w") as f:
    f.write(server_code)

print("wrote notebook_server.py")

## 2. Connect a client and exercise all three primitives

See `docs/09-Building-an-MCP-Client.md`.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = StdioServerParameters(command="python3", args=["notebook_server.py"])

async with stdio_client(SERVER) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        tools = await session.list_tools()
        print("Tools:", [t.name for t in tools.tools])
        result = await session.call_tool("get_disk_usage", {"hostname": "db-primary-01"})
        print("Tool result:", result.content[0].text)

        resources = await session.list_resources()
        print("\nResources:", [str(r.uri) for r in resources.resources])

        prompts = await session.list_prompts()
        rendered = await session.get_prompt("incident_summary", {"log_text": "pod crashed 3 times"})
        print("\nRendered prompt:", rendered.messages[0].content.text)

## 3. Structured errors, not raw tracebacks

See `docs/12-Error-Handling-in-MCP-Tools.md`.

In [ ]:
async with stdio_client(SERVER) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        bad = await session.call_tool("get_disk_usage", {"hostname": "made-up-host"})
        print(bad.content[0].text)

## 4. Auto-derived schemas: vague vs. precise

See `docs/05-Tools.md` and `docs/11-Schema-Design-for-MCP-Tools.md`.
A `FastMCP` object can be introspected in-process - no client
connection needed for this one.

In [ ]:
import json
from typing import Literal
from mcp.server.fastmcp import FastMCP
from pydantic import BaseModel, Field

schema_demo = FastMCP("schema-demo")

@schema_demo.tool()
def vague_tool(x: str, y: str, z: dict) -> dict:
    "Does a thing."
    return {}

class IncidentUpdate(BaseModel):
    incident_id: str = Field(description="The incident ID, e.g. INC-2345")
    status: Literal["open", "investigating", "resolved", "closed"] = Field(description="New status")

@schema_demo.tool()
def precise_tool(update: IncidentUpdate) -> dict:
    "Update an incident's status."
    return {}

for tool in await schema_demo.list_tools():
    print(f"--- {tool.name} ---")
    print(json.dumps(tool.inputSchema, indent=2)[:400])
    print()

## 5. Bridge MCP tools into a local Ollama model

The server has zero knowledge of Ollama - this bridging function is
the only model-specific code involved. See
`docs/10-Bridging-MCP-to-a-Models-Tool-Calling-API.md`.

In [ ]:
import json
from openai import OpenAI

llm_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

def mcp_tool_to_openai_schema(tool):
    return {"type": "function", "function": {"name": tool.name, "description": tool.description or "", "parameters": tool.inputSchema}}

async with stdio_client(SERVER) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        mcp_tools = (await session.list_tools()).tools
        openai_tools = [mcp_tool_to_openai_schema(t) for t in mcp_tools]

        messages = [{"role": "user", "content": "Is disk usage on db-primary-01 critical?"}]
        response = llm_client.chat.completions.create(model="llama3.1:8b", messages=messages, tools=openai_tools)
        msg = response.choices[0].message

        if msg.tool_calls:
            for call in msg.tool_calls:
                args = json.loads(call.function.arguments)
                result = await session.call_tool(call.function.name, args)
                print(f"{call.function.name}({args}) -> {result.content[0].text}")

## 6. Sampling: the server asks the client for a completion

See `docs/15-Sampling-Servers-Requesting-Completions.md`. This
requires a tool that calls `ctx.session.create_message()` - added to a
fresh server file below - and a client-side `sampling_callback` that
fulfills it using the same local Ollama model.

In [ ]:
sampling_server_code = '''
import mcp.types as types
from mcp.server.fastmcp import FastMCP, Context

mcp = FastMCP("sampling-demo")

@mcp.tool()
async def summarize_via_client(text: str, ctx: Context) -> str:
    "Ask the client's model to summarize text."
    result = await ctx.session.create_message(
        messages=[types.SamplingMessage(role="user", content=types.TextContent(type="text", text=f"Summarize in 5 words: {text}"))],
        max_tokens=50,
    )
    return result.content.text

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
with open("notebook_sampling_server.py", "w") as f:
    f.write(sampling_server_code)

import mcp.types as types

async def handle_sampling(context, params):
    messages = [{"role": m.role, "content": m.content.text} for m in params.messages]
    response = llm_client.chat.completions.create(model="llama3.1:8b", messages=messages, max_tokens=params.maxTokens or 50)
    return types.CreateMessageResult(role="assistant", content=types.TextContent(type="text", text=response.choices[0].message.content), model="llama3.1:8b")

SAMPLING_SERVER = StdioServerParameters(command="python3", args=["notebook_sampling_server.py"])

async with stdio_client(SAMPLING_SERVER) as (read, write):
    async with ClientSession(read, write, sampling_callback=handle_sampling) as session:
        await session.initialize()
        result = await session.call_tool("summarize_via_client", {"text": "The service ran out of memory after a slow leak and crashed."})
        print("Server asked, client's model answered:", result.content[0].text)

## 7. Discovery: summarize any connected server generically

See `docs/16-Discovery-Listing-Capabilities.md`.

In [ ]:
async def summarize_server(session):
    tools = await session.list_tools()
    resources = await session.list_resources()
    prompts = await session.list_prompts()
    return {
        "tools": [t.name for t in tools.tools],
        "resources": [str(r.uri) for r in resources.resources],
        "prompts": [p.name for p in prompts.prompts],
    }

async with stdio_client(SERVER) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        print(json.dumps(await summarize_server(session), indent=2))

## 8. A documented gotcha: shared async fixtures

See `docs/17-Testing-an-MCP-Server.md`. This isn't shown as runnable
code here (it's a pytest-specific failure, not something that
reproduces the same way in a notebook cell) - but it's worth reading
the linked chapter before writing MCP tests: a `@pytest.fixture`
wrapping `stdio_client` produces intermittent teardown errors across
test boundaries. Connect inside each test individually instead - see
`../examples/10_test_server.py` for the verified-working pattern.

## Wrap-up

| Section | Concept | Docs chapter |
| --- | --- | --- |
| 1-2 | Server + client basics | 05-09 |
| 3 | Error handling | 12 |
| 4 | Schema design | 05, 11 |
| 5 | Bridging to a local model | 10 |
| 6 | Sampling | 15 |
| 7 | Discovery | 16 |
| 8 | Testing pitfall | 17 |

Not covered here but worth running from `../examples/`:
`04_resource_templates.py` (parameterized resources) and
`07_authorization.py` (a role-based guard clause inside a tool).

Next: `07-AI-Agents` - MCP servers become the standardized tool layer
an autonomous agent reasons over.